In [13]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch
from torchvision import transforms, datasets, models
from gdown import download
from zipfile import ZipFile
from PIL import Image

# 1. Κατεβάζουμε όλες τις εικόνες

In [14]:
def download_and_extract(id, filename, extracted_folder):
    # Κατεβάστε το αρχείο zip εάν δεν υπάρχει ήδη
    if not os.path.exists(filename):
        print("Γίνεται φόρτωση του αρχείου",filename)
        download(id=id, output=filename, quiet=False)

    # Εξαγωγή του αρχείου zip εάν ο φάκελος δεν υπάρχει ήδη
    if not os.path.exists(extracted_folder):
        print("Γίνεται αποσυμπίεση των αρχείων...")
        with ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(extracted_folder)
main_folder = "deepfakes"
download_and_extract('16vuZcaQbd6Mmr4QZgKHJ_Y9UolOef12z', "deepfakes.zip", main_folder)
download_and_extract('1c72PL5KtqOn2wn0wfL4xwmisNPKMKILF', "test.zip", 'test')

# 2. Δημιουργούμε τα Datasets

Ορίζουμε τα train και validation Dataset και τους αντίστοιχους DataLoader

In [15]:
train_folder = os.path.join(main_folder, "train")
validation_folder = os.path.join(main_folder, "validation")

train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=train_folder, transform=train_transform)
validation_dataset = datasets.ImageFolder(root=validation_folder, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
validation_loader = torch.utils.data.DataLoader(validation_dataset, batch_size=32, shuffle=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(validation_dataset)}")

Train dataset size: 40000
Validation dataset size: 2000


# 3. Ορίζουμε το μοντέλο προς εκπαίδευση

**ΣΗΜΑΝΤΙΚΟ!!**

Για την άσκηση πρέπει να χρησιμοποιήσετε υποχρεωτικά το μοντέλο `shufflenet_v2_x1_0` από τα μοντέλα της `torchvision`

In [16]:
model = models.shufflenet_v2_x1_0(num_classes = 2)

if True:
    # Προαιρετικά: το αρχικοποιούμε προ-εκπαιδευμένο με βάση το ImageNet
    model = models.shufflenet_v2_x1_0(weights = models.ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1)
    # Καθώς το imagenet έχει 1000 κατηγορίες τις αλλάζουμε σε 2
    model.fc = torch.nn.Linear(model.fc.in_features, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.train()
model.to(device)

ShuffleNetV2(
  (conv1): Sequential(
    (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (stage2): Sequential(
    (0): InvertedResidual(
      (branch1): Sequential(
        (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=24, bias=False)
        (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): Conv2d(24, 58, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (3): BatchNorm2d(58, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (4): ReLU(inplace=True)
      )
      (branch2): Sequential(
        (0): Conv2d(24, 58, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(58, eps=1e-05, momentum=0.1, affine=True, track_running_

# 4. Εκπαιδεύστε το μοντέλο

Για μεγαλύτερη ταχύτητα επιλέξτε στο Colab τη χρήση GPU π.χ. T4

In [17]:
import copy

def train(model, epochs, optimizer, train_loader, test_loader, criterion, device):
    best_accuracy = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())  # guard: ensure always initialized
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device, non_blocking=True), batch_y.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0   
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device, non_blocking=True)
                batch_y = batch_y.to(device, non_blocking=True)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        scheduler.step()
        val_loss = val_loss / len(test_loader) # Replace with len(val_loader)
        val_accuracy = correct / total
        
        # --- Save Best Model ---
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_model_wts = copy.deepcopy(model.state_dict())
    
        lr = scheduler.get_last_lr()[0]
        print(f"Epoch: {epoch+1}/{epochs}\tTrain Loss: {train_loss:.4f}\tVal Loss: {val_loss:.4f}\tVal Acc: {val_accuracy:.4f}\tLR: {lr:.2e}")    
        model.load_state_dict(best_model_wts)
    print(f"Training complete. Best Validation Accuracy: {best_accuracy:.4f}")

In [18]:
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()))
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [19]:
train(model, epochs, optimizer, train_loader, validation_loader, criterion, device)

KeyboardInterrupt: 

# 5. Εξαγωγή των απαντήσεων

In [ ]:
from glob import glob
import json
answers = {
    "public-sm": "",
    "public-lg": "",
    "private-sm": "",
    "private-lg": ""
}
model.eval()
for key in answers:
    answer = []
    for image in glob("*.jpg", root_dir=os.path.join("test", key)):
        img = Image.open(os.path.join("test", key, image))
        x = train_transform(img)
        x = x.unsqueeze(0).to(device)
        label = model(x).argmax()
        if label == 1: answer.append(image)
    answers[key] = '-'.join(answer)
with open("answers.json", "w") as f:
    json.dump(answers, f)